# FunderWonder

---
## Setup

Run the cell below to install the required packages. You may see some warnings or dependency messages; these are safe to ignore.

In [ ]:
 # Install required packages (this may take a minute, and you may ignore the errors)
!pip install -qU langchain-google-genai langchain-community langchain-experimental google-search-results
!pip -q install google-api-python-client google-auth google-auth-httplib2 google-auth-oauthlib

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.


In [ ]:
# Configure your API keys
# You should already have GOOGLE_API_KEY saved in your Colab secrets.

import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["GCP_CREDENTIALS"] = userdata.get("GCP_CREDENTIALS")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
print("API keys configured successfully!")

API keys configured successfully!


### Step 1: Create the LLM

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

---
## Part A: Implement Grants.Gov API Tool + Tavily Search


In [ ]:
import requests
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults

tavily_search = TavilySearchResults(max_results=3)

# Common headers to avoid being blocked by government APIs
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

@tool
def search_grants(keywords: str) -> str:
    """
    Searches Grants.gov for open and forecasted grants.
    Use this to find a list of potential grant opportunities.
    """
    url = "https://api.grants.gov/v1/api/search2"

    payload = {
        "keyword": keywords,
        # "oppStatuses": "posted|forecasted", // testing only posted grants
        "oppStatuses": "posted",
        "rows": 10
    }

    try:
        response = requests.post(url, json=payload, headers=HEADERS)

        if response.status_code == 200:
            results = response.json()

            # Extract opportunities from nested structure
            data = results.get("data", {})
            opportunities = data.get("oppHits") or []

            output = ""
            for opp in opportunities:
                output += f"ID: {opp.get('id')} | Number: {opp.get('number')} | Title: {opp.get('title')}\n"

            return output if output else "No grants found for these keywords."
        else:
            return f"Error: Received status code {response.status_code}"
    except Exception as e:
        return f"Error searching grants: {str(e)}"


@tool
def get_grant_details(opportunity_id: str) -> str:
    """
    Fetches the full description and synopsis for a specific grant ID.
    If the official description is missing, it falls back to a web search.
    """
    clean_id = str(opportunity_id).replace("ID:", "").strip()
    url = "https://api.grants.gov/v1/api/fetchOpportunity"
    payload = {"opportunityId": clean_id}

    try:
        response = requests.post(url, json=payload, headers=HEADERS)

        if response.status_code == 200:
            response_json = response.json()
            data = response_json.get("data", {})

            # Extract basic info for a potential fallback search
            title = data.get("opportunityTitle", "Unknown Title")
            number = data.get("fundingOpportunityNumber", "Unknown Number")

            # 1. Check for Synopsis (Posted Grants)
            synopsis = data.get("synopsis")
            if synopsis:
                desc = synopsis.get("synopsisExplanation") or synopsis.get("synopsisDesc")
                if desc:
                    return f"Official Synopsis: {desc}"

            # 2. Check for Forecast (Forecasted Grants)
            forecast = data.get("forecast")
            if forecast:
                desc = forecast.get("forecastExplanation") or forecast.get("forecastDesc")
                if desc:
                    return f"Official Forecast: {desc}"

            # 3. TAVILY FALLBACK
            # If we get here, the government database is empty. Let's search the web!
            print(f"No official description found for {number}. Falling back to Tavily web search...")
            search_query = f"Grants.gov {number} {title} grant summary description eligibility"
            web_results = tavily_search.invoke({"query": search_query})

            return f"No official API description available. Web Search Fallback Results for {number}:\n{web_results}"

        else:
            return f"Error: Could not fetch details for ID {clean_id}."
    except Exception as e:
        return f"Error fetching details: {str(e)}"

@tool
def score_grant_match(grant_description: str, user_profile: str, project_needs: str) -> str:
    """
    Calculates a FunderWonder Match Score (0-100) between a grant and a user's specific needs.
    Use this AFTER fetching the grant details to advise the user on whether they should apply.
    """
    scoring_prompt = f"""
    You are an expert grant evaluator. Evaluate this grant out of 100 based on the user's profile and needs.

    User Profile: {user_profile}
    Project Needs: {project_needs}
    Grant Description: {grant_description}

    Calculate the score using this rubric:
    - Relevance (40 pts): Does the research topic align with their project needs?
    - Eligibility (30 pts): Can this specific user (based on their profile) apply?
    - Funding Type (30 pts): Does it cover what they need?

    Format your response as:
    **FunderWonder Match Score: [Score]/100**
    * **Relevance:** [Brief reason]
    * **Eligibility:** [Brief reason]
    * **Funding Type:** [Brief reason]
    """

    response = llm.invoke(scoring_prompt)
    return response.content

@tool
def search_grants_with_details(keywords: str) -> str:
    """
    Searches for grants AND automatically fetches details for each one.
    Returns a combined summary with both the grant info and its description.
    Use this when the user wants a complete overview or a document.
    """
    url = "https://api.grants.gov/v1/api/search2"
    payload = {
        "keyword": keywords,
        "oppStatuses": "posted|forecasted",
        "rows": 5
    }

    try:
        response = requests.post(url, json=payload, headers=HEADERS)
        if response.status_code != 200:
            return f"Error: Received status code {response.status_code}"

        results = response.json()
        data = results.get("data", {})
        opportunities = data.get("oppHits") or []

        if not opportunities:
            return "No grants found for these keywords."

        # Loop through each grant and fetch its details
        combined_output = ""
        for i, opp in enumerate(opportunities, 1):
            grant_id = opp.get('id')
            title = opp.get('title', 'Unknown Title')
            number = opp.get('number', 'N/A')

            combined_output += f"\n{'='*60}\n"
            combined_output += f"GRANT {i}: {title}\n"
            combined_output += f"Number: {number} | ID: {grant_id}\n"
            combined_output += f"{'='*60}\n"

            # Fetch details for this specific grant
            details = get_grant_details.invoke(str(grant_id))
            combined_output += f"Description: {details}\n"

        return combined_output

    except Exception as e:
        return f"Error: {str(e)}"

@tool
def generate_and_save_proposal(opportunity_id: str, user_profile: str, project_description: str) -> str:
    """
    Generates a proposal AND saves it directly to Google Docs.
    """

    grant_details = get_grant_details.invoke(opportunity_id)

    proposal_prompt = f"""
    You are an expert grant proposal writer for students trying to find funding.

    USER PROFILE:
    {user_profile}

    PROJECT DESCRIPTION:
    {project_description}

    GRANT DETAILS:
    {grant_details}

    Write a full professional grant proposal.
    """

    proposal = llm.invoke(proposal_prompt).content

    #Saving to Google Docs
    doc_result = create_proposal_doc.invoke({
        "title": "Grant Proposal Draft",
        "content": proposal
    })

    return f"{doc_result}\n\n---\n\nPreview:\n{proposal[:1000]}..."



/tmp/ipykernel_10339/3696879287.py:5: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_search = TavilySearchResults(max_results=3)


---
## Part B: Authorize Google Drive/Google Docs

In [ ]:
import json
from google_auth_oauthlib.flow import Flow
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request

# Define the scopes required
SCOPES = [
    'https://www.googleapis.com/auth/documents',
    'https://www.googleapis.com/auth/drive.file'
]

creds = None

# 1. Check if we already have a saved token from a previous run
if os.path.exists('token.json'):
    creds = Credentials.from_authorized_user_file('token.json', SCOPES)

# 2. If no valid credentials exist, run the auth flow
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        # Token is expired but we can refresh it silently
        creds.refresh(Request())
    else:
        # No token at all, do the manual OAuth
        secret_value = userdata.get('GCP_CREDENTIALS')
        with open('credentials.json', 'w') as f:
            f.write(secret_value)

        flow = Flow.from_client_secrets_file(
            'credentials.json',
            scopes=SCOPES,
            redirect_uri='urn:ietf:wg:oauth:2.0:oob'
        )

        auth_url, _ = flow.authorization_url(prompt='consent')

        print(f"1. Visit this URL to authorize: {auth_url}")
        print("\n2. Sign in and click 'Allow'.")
        print("3. You will see a page with a code. Copy it.")

        code = input("\n4. Enter the authorization code here: ")

        flow.fetch_token(code=code)
        creds = flow.credentials

    # Save the fresh credentials for the next time you run this cell
    with open('token.json', 'w') as token:
        token.write(creds.to_json())

print("\nAuthentication successful! Your agent is ready to write to Google Docs.")

1. Visit this URL to authorize: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=584926836345-rt23bmn5p3o2jamehuol812qameqn9bn.apps.googleusercontent.com&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdocuments+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.file&state=aQ9pyKILZJNmtl3BCEoW17Y0ku4Sry&code_challenge=JY-udwSYGmBas6NgfRTZL4Bo1ptO5YP32FjWM4Vrq_c&code_challenge_method=S256&prompt=consent&access_type=offline

2. Sign in and click 'Allow'.
3. You will see a page with a code. Copy it.

4. Enter the authorization code here: 4/1Aci98E_Rslxi3TafDeJ8WV34OFbWwc6PVuSFYHz_x17xNQLDq49Lo1DFIZg

Authentication successful! Your agent is ready to write to Google Docs.


---
## Part C: Implement Google Docs API Tool

In [ ]:
from googleapiclient.discovery import build

# Build the Docs service using your active credentials
docs_service = build('docs', 'v1', credentials=creds)

@tool
def create_proposal_doc(title: str, content: str) -> str:
    """
    Creates a new Google Doc with the specified title and writes the content into it.
    Returns the URL of the created document.
    """
    try:
        # Step 1: Create a blank document
        doc = docs_service.documents().create(body={'title': title}).execute()
        doc_id = doc.get('documentId')

        # Step 2: Insert the text into the document
        # We use index 1 because index 0 is the start of the document structure
        requests = [
            {
                'insertText': {
                    'location': {'index': 1},
                    'text': content
                }
            }
        ]
        docs_service.documents().batchUpdate(documentId=doc_id, body={'requests': requests}).execute()

        return f"Document created successfully! URL: https://docs.google.com/document/d/{doc_id}/edit"
    except Exception as e:
        return f"Error creating Google Doc: {str(e)}"

In [ ]:
agent_prompt = """You are FunderWonder, an expert research assistant specializing in finding and securing grant funding.

You have access to the following tools. NEVER hallucinate tool names.
1. search_grants(keywords): Searches for grants and returns summaries with their numeric IDs.
2. get_grant_details(opportunity_id): Fetches the full description of a specific grant.
3. score_grant_match(grant_description, user_profile, project_needs): Calculates a match score.
4. generate_and_save_proposal(opportunity_id, user_profile, project_description): Drafts a full proposal and automatically saves it as a Google Doc.

Follow this strict workflow based on the user's request:
- Phase 1: Discovery. If the user wants to find grants, ALWAYS use `search_grants` first. Present the findings clearly with their numeric IDs.
- Phase 2: Evaluation. If the user asks if a specific grant is a "good fit", you must ensure you have BOTH their personal/professional background AND their project needs. If you are missing either, ASK the user for them before proceeding. Once you have the info, run `get_grant_details` and then immediately run `score_grant_match`.
- Phase 3: Proposal Generation. If the user asks to write a proposal, use `generate_and_save_proposal`. Since you already gathered their profile and project details in Phase 2, you can pass them directly into this tool.
"""

grant_agent = create_agent(
    model=llm,
    tools=[search_grants, get_grant_details, score_grant_match, create_proposal_doc, generate_and_save_proposal],
    system_prompt=agent_prompt
)

In [ ]:
chat_history = []

def clear_memory():
    global chat_history
    chat_history = []
    print("\n--- Memory Cleared! ---\n")

def run_agent_with_memory(user_input):
    # 1. Add the user's new message to the persistent history
    chat_history.append({"role": "human", "content": user_input})

    print(f"\n--- USER: {user_input} ---\n")

    # 2. Pass the ENTIRE history to the agent so it has context
    for chunk in grant_agent.stream(
        {"messages": chat_history},
        stream_mode="updates"
    ):
        for step, data in chunk.items():
            print(f"Step: {step}")

            # 3. Add the agent's responses (and tool outputs) back to the history
            new_messages = data['messages']
            for msg in new_messages:
                chat_history.append(msg)

                if hasattr(msg, 'content_blocks'):
                    print(f"Content: {msg.content_blocks}")
                else:
                    print(f"Content: {msg.content}")
            print()

# Uncomment the line below to reset the memory before testing!
clear_memory()

# Test it out! Notice how we split the interaction into two steps now.
run_agent_with_memory("I am an individual grad student needing $2000 for cancer research. Score grant ID 360918.")

# Run the follow-up. The agent will now remember the previous context!
run_agent_with_memory("My background is in CS and AI. My project is an NLP matching platform.")


--- USER: I am an individual grad student needing $2000 for cancer research. Score grant ID 360918. ---

Step: model
Content: [{'type': 'text', 'text': 'Okay, I can help you evaluate grant ID 360918. But first, I need a bit more information to calculate the match score accurately. Could you please provide:\n\n1.  **Your User Profile:** Tell me about your background, including your field of study, research experience, and any relevant qualifications.\n2.  **Your Project Needs:** Describe how you plan to use the \\$2000 for your cancer research. Be specific about the resources, equipment, or support you require.\n\nOnce I have this information, I will fetch the grant details and then calculate the match score.'}]


--- USER: My background is in CS and AI. My project is an NLP matching platform. ---

Step: model
Content: [{'type': 'text', 'text': 'Okay, I have your user profile and project needs. Now I will get the grant details for grant ID 360918 and then calculate the match score.'}, 

In [ ]:
test_query = "Find cancer research grants and summarize them"

for chunk in grant_agent.stream(
    {"messages": [{"role": "human", "content": test_query}]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"Step: {step}")
        print(f"Content: {data['messages'][-1].content_blocks}")
        print()

Step: model
Content: [{'type': 'tool_call', 'id': '66447066-3f6a-4e69-925d-8c37362219fe', 'name': 'search_grants', 'args': {'keywords': 'cancer research'}}]

Step: tools
Content: [{'type': 'text', 'text': "ID: 360918 | Number: PAR-25-444 | Title: Cancer Center Support Grants (CCSGs) for NCI-designated Cancer Centers (P30 Clinical Trial Optional)\nID: 350299 | Number: PAR-23-284 | Title: Specialized Programs of Research Excellence (SPOREs) in Human Cancers for Years 2024, 2025, and 2026 (P50 Clinical Trial Required)\nID: 357192 | Number: PAR-25-243 | Title: Basic Research in Cancer Health Disparities (R01 Clinical Trial Not Allowed)\nID: 357193 | Number: PAR-25-244 | Title: Basic Research in Cancer Health Disparities (R21 Clinical Trial Not Allowed)\nID: 357075 | Number: PAR-25-081 | Title: National Cancer Institute's Investigator-Initiated Early Phase Clinical Trials for Cancer Treatment and Diagnosis (R01 Clinical Trial Required)\nID: 357304 | Number: PAR-25-254 | Title: Understanding

In [ ]:
test_query = "I am an individual grad student needing $2000 for cancer research. Score grant ID 360918."

for chunk in grant_agent.stream(
    {"messages": [{"role": "human", "content": test_query}]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"Step: {step}")
        print(f"Content: {data['messages'][-1].content_blocks}")
        print()

Step: model
Content: [{'type': 'text', 'text': "Okay, I can help you evaluate grant ID 360918. But first, I need some information to calculate a match score:\n\n*   **Your Profile:** Tell me about your background as a grad student. What's your field of study, university affiliation, and any relevant experience?\n*   **Project Needs:** Describe your cancer research project and how the $2000 would be used. Be specific about the resources, equipment, or support you require.\n\nOnce I have this information, I will fetch the grant details and then calculate the match score."}]



In [ ]:
test_query = "Can you explain what grant ID 359855 is?"

for chunk in grant_agent.stream(
    {"messages": [{"role": "human", "content": test_query}]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"Step: {step}")
        print(f"Content: {data['messages'][-1].content_blocks}")
        print()

Step: model
Content: [{'type': 'tool_call', 'id': 'd47ceacd-85f4-4f3d-b119-3f9de17fa3db', 'name': 'get_grant_details', 'args': {'opportunity_id': '359855'}}]

Step: tools
Content: [{'type': 'text', 'text': 'Official Forecast: <p>The National Cancer Institute intends to publish a Notice of Funding Opportunity (NOFO) to invite&nbsp;applications for&nbsp;advanced development and enhancement of emerging informatics technologies&nbsp;to improve the acquisition, analysis, visualization, and interpretation of data across the cancer research continuum including cancer biology, cancer treatment and diagnosis, early cancer detection, risk assessment and prevention, cancer control and epidemiology, and cancer health disparities. As a component of the NCI\'s&nbsp;<a href="http://itcr.cancer.gov/" target="_blank" style="color: rgb(31, 107, 178);">Informatics Technology for Cancer Research</a>&nbsp;(ITCR) Program, this NOFO will focus on&nbsp;advancing emerging informatics technology, defined as one

In [ ]:
test_query = """
I want to see if Grant ID 359855 is a good fit.

My background:
CS student focused on AI and backend systems.

My project:
An AI grant matching platform using NLP.
"""

for chunk in grant_agent.stream(
    {"messages": [{"role": "human", "content": test_query}]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"Step: {step}")
        print(f"Content: {data['messages'][-1].content_blocks}")
        print()

Step: model
Content: [{'type': 'text', 'text': "Okay, I can help you with that. To determine if Grant ID 359855 is a good fit for you, I need to fetch the grant details and then score the match based on your profile and project needs.\n\nFirst, I need to get the grant details for Grant ID 359855. After that, I'll use your background as a CS student focused on AI and backend systems, and your project of building an AI grant matching platform using NLP to calculate a match score."}, {'type': 'tool_call', 'id': 'fe321753-3b56-4209-ba4b-dc225e8134fc', 'name': 'get_grant_details', 'args': {'opportunity_id': '359855'}}]

Step: tools
Content: [{'type': 'text', 'text': 'Official Forecast: <p>The National Cancer Institute intends to publish a Notice of Funding Opportunity (NOFO) to invite&nbsp;applications for&nbsp;advanced development and enhancement of emerging informatics technologies&nbsp;to improve the acquisition, analysis, visualization, and interpretation of data across the cancer resea

In [ ]:
test_query = """
I want to apply for grant ID 359855.

Project Title: AI-Powered Grant Recommendation System

My background:
CS student focused on AI and backend systems.

My project:
An AI grant matching platform using NLP.

Generate a proposal AND save it as a Google Doc.
"""
for chunk in grant_agent.stream(
    {"messages": [{"role": "human", "content": test_query}]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"Step: {step}")
        print(f"Content: {data['messages'][-1].content_blocks}")
        print()

Step: model
Content: [{'type': 'tool_call', 'id': 'c24fc8ef-676f-4252-8baa-635b65bd2c29', 'name': 'generate_and_save_proposal', 'args': {'project_description': 'An AI grant matching platform using NLP.', 'user_profile': 'CS student focused on AI and backend systems.', 'opportunity_id': '359855'}}]

Step: tools
Content: [{'type': 'text', 'text': "Document created successfully! URL: https://docs.google.com/document/d/1d8ZFj55IFlfpcA8o9fZKeMZ8dm95HRM75p61_LXaIlo/edit\n\n---\n\nPreview:\nOkay, here's a grant proposal draft tailored to the NCI's ITCR U24 NOFO, focusing on your AI-powered grant matching platform. This proposal emphasizes the AI and backend systems aspects while highlighting the potential impact on cancer research. Note that this is a *draft* and needs to be refined with specific details, data, and collaborations as you develop your project.  Also, remember applications are not being solicited at this time. This is to allow you time to develop collaborations and projects.\n\n

In [ ]:
# Install Flask and Cloudflare tunnel so your frontend can reach Colab
!pip install -q flask flask-cors flask-cloudflared

from flask import Flask, request, jsonify
from flask_cors import CORS
from flask_cloudflared import run_with_cloudflared

app = Flask(__name__)
CORS(app) # Allows your HTML file to communicate with this server
run_with_cloudflared(app) # Creates a public URL for your Colab cell

@app.route('/chat', methods=['POST'])
def chat():
    data = request.json

    # 1. Extract the history sent from the frontend's JavaScript
    client_history = data.get('history', [])

    try:
        # 2. Pass the frontend's history directly into the LangChain agent
        response = grant_agent.invoke({"messages": client_history})

        # 3. Extract the final text response
        final_text = response["messages"][-1].content

        return jsonify({"response": final_text})

    except Exception as e:
        return jsonify({"error": str(e)}), 500

# Start the server!
if __name__ == '__main__':
    app.run()